# Load Static Embeddings

Run the cells below to download the static embeddings used by the classic baseline experiments.

## GloVe

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

GLOVE_DIR = Path("experiments/embeddings")
GLOVE_ZIP = GLOVE_DIR / "glove.6B.zip"
GLOVE_PATH = GLOVE_DIR / "glove.6B.300d.txt"
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"

GLOVE_DIR.mkdir(parents=True, exist_ok=True)

if not GLOVE_PATH.exists():
    if not GLOVE_ZIP.exists():
        print("Downloading GloVe. This file is large, so it may take a while.")
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as zf:
        zf.extract(GLOVE_PATH.name, path=GLOVE_DIR)

print(GLOVE_PATH)

## fastText

In [ ]:
from pathlib import Path
import urllib.request

FASTTEXT_DIR = Path("experiments/embeddings")
FASTTEXT_EN_PATH = FASTTEXT_DIR / "fasttext_en.vec"
FASTTEXT_DE_PATH = FASTTEXT_DIR / "fasttext_de.vec"
FASTTEXT_PATH = FASTTEXT_DIR / "fasttext.vec"

FASTTEXT_EN_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-aligned/wiki.en.align.vec"
FASTTEXT_DE_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-aligned/wiki.de.align.vec"

FASTTEXT_DIR.mkdir(parents=True, exist_ok=True)

def read_fasttext_header(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        count, dim = f.readline().strip().split()
    return int(count), int(dim)

def iter_fasttext_vectors(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        next(f)
        for line in f:
            token = line.split(" ", 1)[0]
            yield token, line

def merge_aligned_fasttext(english_path, german_path, merged_path):
    _, english_dim = read_fasttext_header(english_path)
    _, german_dim = read_fasttext_header(german_path)
    if english_dim != german_dim:
        raise ValueError(f"Dimension mismatch: {english_dim} != {german_dim}")

    seen = set()
    total = 0
    # German is read first so duplicate surface forms keep the German vector.
    for path in [german_path, english_path]:
        for token, _ in iter_fasttext_vectors(path):
            if token not in seen:
                seen.add(token)
                total += 1

    seen.clear()
    with merged_path.open("w", encoding="utf-8", newline="\n") as out:
        out.write(f"{total} {english_dim}\n")
        # German is written first so duplicate surface forms keep the German vector.
        for path in [german_path, english_path]:
            for token, line in iter_fasttext_vectors(path):
                if token not in seen:
                    seen.add(token)
                    out.write(line)

if not FASTTEXT_EN_PATH.exists():
    print("Downloading aligned English fastText vectors. This file is large, so it may take a while.")
    urllib.request.urlretrieve(FASTTEXT_EN_URL, FASTTEXT_EN_PATH)

if not FASTTEXT_DE_PATH.exists():
    print("Downloading aligned German fastText vectors. This file is large, so it may take a while.")
    urllib.request.urlretrieve(FASTTEXT_DE_URL, FASTTEXT_DE_PATH)

if not FASTTEXT_PATH.exists():
    print("Merging aligned English and German fastText vectors.")
    merge_aligned_fasttext(FASTTEXT_EN_PATH, FASTTEXT_DE_PATH, FASTTEXT_PATH)

print(FASTTEXT_PATH)